# 한국어 TTS

- OmniVoice 는 한국어를 지원함 (600 + 언어 중 하나)
- 별도 언어 인자 없이 한국어 텍스트를 그대로 넘기면 됩니다.

## 1. 모델 로드 (이미 캐시되어 있으면 빠름)

In [1]:
# 라이브러리 다운로드
%pip install omnivoice

Note: you may need to restart the kernel to use updated packages.


- Hugging Face 토큰 에러 발생시 아래 코드 실행

In [2]:
import os

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
try:
    import huggingface_hub.constants as hf_constants
    hf_constants.HF_HUB_DISABLE_IMPLICIT_TOKEN = True
except Exception:
    pass

In [3]:
import torch
import soundfile as sf
from omnivoice import OmniVoice
from IPython.display import Audio
import os
os.makedirs("outputs", exist_ok=True)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32

model = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map=device, dtype=dtype)
print("준비 완료. device:", device)

c:\Users\playdata2\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).
W0709 11:34:47.119000 12136 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 527/527 [00:00<00:00, 6946.57it/s]

준비 완료. device: cpu


## 2. 한국어 합성, 짧은 인사

In [5]:
audio = model.generate(text='안녕하세요. 꾸러기 학생 여러분, 반갑습니다.')
sf.write('./outputs/ko_greeting.wav', audio[0], 24000)
Audio('./outputs/ko_greeting.wav')

## 3. 다양한 문장, 콘텐츠 시나리오

게임 NPC 대사, 뉴스 톤, 친근한 인사 같은 시나리오 별로 합성해서 결과 비교.

In [6]:
scenarios = {
    "ko_npc_quest"   : "용사여, 다음 던전으로 가는 길은 북쪽 숲을 지나야 합니다.",
    "ko_news_intro"  : "오늘의 인공지능 헤드라인 소식부터 전해드리겠습니다.",
    "ko_friendly"    : "오늘 점심 같이 먹을래? 내 방에서 라면 먹고 갈래?",
    "ko_boss_warning": "감히 내 영역에 발을 들이다니. 후회하게 만들어주마.",
}

for name, text in scenarios.items():
    audio = model.generate(text=text)
    path = f'outputs/{name}.wav'
    sf.write(path, audio[0], 24000)
    print(f"  saved: {path}  ({len(audio[0])/24000:.1f}초)")

  saved: outputs/ko_npc_quest.wav  (4.1초)
  saved: outputs/ko_news_intro.wav  (3.5초)
  saved: outputs/ko_friendly.wav  (3.0초)
  saved: outputs/ko_boss_warning.wav  (3.7초)


### 결과 한 번에 재생

In [7]:
for name in scenarios:
    print(name)
    display(Audio(f'outputs/{name}.wav'))

ko_npc_quest


ko_news_intro


ko_friendly


ko_boss_warning


## 4. 긴 문단도 한 번에

문장 부호로 자연스럽게 호흡이 들어갑니다. 너무 긴 문단은 끊어서 합성한 뒤 이어 붙이는 게 안전.

In [8]:
long_text = (
    "sk네트웍스 Family AI camp에 오신 것을 환영합니다. "
    "오늘 우리는 인공지능 음성 합성 기술을 다루는 시간을 가질 예정입니다. "
    "차근차근 따라오시면 누구나 자신만의 목소리 모델을 만들 수 있습니다."
)

audio = model.generate(text=long_text)
sf.write("./outputs/ko_paragraph.wav", audio[0], 24000)
print(f"길이: {len(audio[0])/24000:.1f}초")
Audio("./outputs/ko_paragraph.wav")

길이: 14.4초


## [실습]
1. 같은 문장을 영어로 한 번, 한국어로 한 번 합성해 발음·억양 차이를 비교.
2. 자기가 만든 게임 캐릭터의 첫 등장 대사를 합성.
3. 3 문장짜리 NPC 시나리오 (만남 -> 퀘스트 -> 작별) 를 합성하고 하나로 이어붙이기.
4. 같은 의미를 여섯 가지 톤 (공손/친근/엄격/장난기/긴장/속삭임) 으로 텍스트를 재작성해 합성, 톤 차이를 정성적으로 비교.